# Protocol Comparison: Criticality Diagnostics for Perfect-Correction vs Post-Selection

**Campaign**: `trace_chargevar_flattened_energy_vs_cycle_protocols`  
**System**: Class A topological insulator, Nx=20, Ny=40 (Nlayer=1600 modes)

## Claim being tested

The `perfect_correction` protocol is **not charge-conserving**: each ancilla-swap correction adds or removes a fermion. The steady state is therefore an ensemble that mixes particle-number sectors, while `postselect` is exactly half-filled in the ideal charge-conserving picture. The two protocols reach different flattened energies, and the goal here is to compare those energies against a **full-spectrum half-filling reference** rather than a DW-only effective model.

**Key physics note**: the central charge of a free chiral fermion is c=1 regardless of filling. A shift in occupation relative to half-filling can change the energy and charge statistics, but by itself does not explain a change in c. Any difference in c between the protocols (seen in separate Cardy-Calabrese entanglement data) must have a different origin.

### Tests performed in this notebook

1. **Flattened-H spectrum**: build the full flattened Hamiltonian spectrum and the cumulative many-body energy curve as a function of occupation relative to half-filling.
2. **Tr(G) histograms**: at multiple cycles, check whether `perfect_correction` samples are distributed around a non-zero mean (charge imbalance).
3. **Charge variance vs cycle**: compare the maximally mixed runs across protocols.
4. **Flattened energy vs cycle**: compare the full energy trajectories for all four runs.
5. **Default-init occupation inference**: map final-sample energies onto the nearest cumulative full-spectrum many-body occupation relative to half-filling.


## Dataset summary

| init_mode | protocol          | samples | cycles | charge_variance saved | CV final | energy final |
|-----------|-------------------|---------|--------|-----------------------|----------|--------------|
| maxmix    | perfect_correction | 100    | 50     | ✓                     | 0.181    | −1255.6      |
| maxmix    | postselect         | 1      | 50     | ✓                     | 0.502    | −1272.8      |
| default   | perfect_correction | 100    | 50     | ✗                     | —        | −1255.9      |
| default   | postselect         | 1      | 50     | ✗                     | —        | −1272.8      |

`maxmix` starts from the maximally mixed state (G=0); `default` starts from the pure Slater-determinant ground state.  
Postselect forces a single trajectory (charge-conserving): `samples=1` is not a statistical limitation — every postselected trajectory stays at Tr(G)=0 exactly.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats as sp_stats

# ── colour scheme (colour-blind safe) ──────────────────────────────────────
PC_COLOR = '#0072B2'   # blue  — perfect_correction
PS_COLOR = '#D55E00'   # orange — postselect
MM_STYLE = dict(linestyle='-',  marker='o', markersize=3)
DF_STYLE = dict(linestyle='--', marker='s', markersize=3)

BUNDLE_NAME = 'colab_small_system_testing'
CAMPAIGN_NAME = 'trace_chargevar_flattened_energy_vs_cycle_protocols'

def is_bundle_root(path: Path) -> bool:
    return (path / 'gpu_data').exists() and (path / 'notebooks').exists()

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])

seen: set = set()
BUNDLE_ROOT: Path | None = None
for _c in candidates:
    _c = _c.resolve()
    if _c in seen:
        continue
    seen.add(_c)
    if is_bundle_root(_c):
        BUNDLE_ROOT = _c
        break

if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate bundle root ({BUNDLE_NAME})')

GPU_DATA_ROOT  = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_ROOT  = GPU_DATA_ROOT / CAMPAIGN_NAME
ANALYSIS_ROOT  = BUNDLE_ROOT / 'analysis_outputs' / CAMPAIGN_NAME
FIGURE_ROOT    = ANALYSIS_ROOT / 'figures'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

# add src/ so we can import classA_U1FGTN
REPO_ROOT = BUNDLE_ROOT.parent
for _src in [REPO_ROOT / 'src', BUNDLE_ROOT / 'src']:
    if _src.exists() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

print(f'[setup] bundle root : {BUNDLE_ROOT}')
print(f'[setup] figure root : {FIGURE_ROOT}')

In [ ]:
# ── Build flattened Hamiltonian and full-spectrum half-filling reference ───
from fgtn.classA_U1FGTN import classA_U1FGTN   # CPU version from main src/

NX, NY = 20, 40
NLAYER = NX * NY * 2  # 1600 modes

model = classA_U1FGTN(Nx=NX, Ny=NY)
# match campaign config: nshell=1, DW=True, trial_orbitals='X', dw_truncation=True
model.construct_OW_projectors(nshell=1, DW=True, trial_orbitals='X', dw_truncation=True)

# Build H_flat = P_Ap + P_Bp - P_Am - P_Bm
WF_Ap = model.WF_Ap.reshape(NLAYER, -1)
WF_Bp = model.WF_Bp.reshape(NLAYER, -1)
WF_Am = model.WF_Am.reshape(NLAYER, -1)
WF_Bm = model.WF_Bm.reshape(NLAYER, -1)

H_flat = (
    WF_Ap @ WF_Ap.T.conj()
    + WF_Bp @ WF_Bp.T.conj()
    - WF_Am @ WF_Am.T.conj()
    - WF_Bm @ WF_Bm.T.conj()
)
H_flat = 0.5 * (H_flat + H_flat.T.conj())  # symmetrise numerically

evals = np.linalg.eigvalsh(H_flat)
evals_sorted = np.sort(np.asarray(evals, dtype=float))

# Full-spectrum many-body occupation ladder:
# n = 0,1,2,... means occupy the n lowest single-particle levels.
n_occ_grid = np.arange(evals_sorted.size + 1, dtype=int)
cum_energy = np.concatenate([[0.0], np.cumsum(evals_sorted)])

# Half-filling = occupy all negative-energy levels
n_half = int(np.count_nonzero(evals_sorted < 0.0))
E_half = float(cum_energy[n_half])
n_zero = int(np.count_nonzero(np.abs(evals_sorted) < 1e-6))

print(f'Nlayer = {NLAYER}')
print(f'spectrum min/max = {evals_sorted[0]:.4f}, {evals_sorted[-1]:.4f}')
print(f'number of exactly-zero modes = {n_zero}')
print(f'n_half = {n_half}')
print(f'E_half = {E_half:.6f}')
print(f'cumulative energy range = [{cum_energy.min():.6f}, {cum_energy.max():.6f}]')


In [ ]:
# ── Load campaign manifest and all NPZ data ────────────────────────────────
latest_campaign_path = CAMPAIGN_ROOT / 'latest_campaign.json'
with latest_campaign_path.open() as fh:
    latest_campaign = json.load(fh)

manifest_path = GPU_DATA_ROOT / latest_campaign['manifest']
with manifest_path.open() as fh:
    manifest = json.load(fh)

run_map: dict = {}
for item in manifest['results']:
    npz_path = GPU_DATA_ROOT / item['npz_path_relative']
    with np.load(npz_path, allow_pickle=False) as d:
        rec = {
            'cycles':                  d['cycles'].copy(),
            'trace_by_sample':         d['trace_real_by_sample'].copy(),
            'trace_mean':              d['trace_real_mean'].copy(),
            'trace_std':               d['trace_real_std'].copy(),
            'trace_stderr':            d['trace_real_stderr'].copy(),
            'energy_by_sample':        d['flattened_energy_by_sample'].copy(),
            'energy_mean':             d['flattened_energy_mean'].copy(),
            'energy_std':              d['flattened_energy_std'].copy(),
            'energy_stderr':           d['flattened_energy_stderr'].copy(),
        }
        if 'charge_variance_by_sample' in d.files:
            rec['cv_by_sample'] = d['charge_variance_by_sample'].copy()
            rec['cv_mean']      = d['charge_variance_mean'].copy()
            rec['cv_std']       = d['charge_variance_std'].copy()
            rec['cv_stderr']    = d['charge_variance_stderr'].copy()
    key = (item['init_mode'], item['protocol'])
    run_map[key] = rec

print(f'[load] campaign id: {manifest["campaign_id"]}')
for k in run_map:
    r = run_map[k]
    n = r['trace_by_sample'].shape[0]
    print(f'  {k}: {n} samples, cycles 0–{r["cycles"][-1]}, '
          f'E_final={r["energy_mean"][-1]:.1f}, '
          f'Tr_mean={r["trace_mean"][-1]:.3f}')

## 1. Flattened-H spectrum and full-spectrum half-filling reference

The flattened Hamiltonian `H_flat` is built from `nshell=1` Onsite-Wannier projectors with `dw_truncation=True`, matching the campaign config. Rather than separating DW and bulk modes, this notebook now treats the **entire flattened spectrum on equal footing**.

The reference construction is:

- sort all single-particle eigenvalues of `H_flat`
- define the many-body occupation count `n = 0, 1, 2, ...` by occupying the `n` lowest levels
- build the cumulative many-body energy `E(n)` as a running sum over the sorted spectrum
- define **half-filling** by occupying all negative-energy levels, giving `n_half` and `E_half`

All later occupation-based comparisons are made relative to this full-spectrum half-filling point. No DW-only thresholding or in-gap mode counting is used in the notebook anymore.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

# ── left: full ordered spectrum ────────────────────────────────────────────
ax = axes[0]
idx = np.arange(len(evals_sorted))
ax.scatter(idx, evals_sorted, s=2.0, c='#666666')
ax.axhline(0.0, color='black', lw=0.8, alpha=0.5)
ax.set_xlabel('mode index (sorted)')
ax.set_ylabel('eigenvalue h')
ax.set_title('Full ordered spectrum of H_flat')
ax.grid(alpha=0.2)

# ── right: cumulative many-body energy relative to half-filling ───────────
ax = axes[1]
delta_N_grid = n_occ_grid - n_half
ax.scatter(delta_N_grid, cum_energy, s=8, c=PC_COLOR, alpha=0.8)
ax.axvline(0, color='grey', lw=0.9, ls=':', alpha=0.7, label='half-filling')
ax.axhline(E_half, color='black', lw=0.9, ls='--', alpha=0.7, label='E_half')
ax.set_xlabel('many-body occupation relative to half-filling  ($\\Delta N$)')
ax.set_ylabel('cumulative many-body energy')
ax.set_title('Full-spectrum cumulative energy ladder')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

fig.suptitle(f'{manifest["campaign_id"]}: H_flat full-spectrum reference (Nx={NX}, Ny={NY})', fontsize=13)

fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_hflat_full_spectrum_reference.png', dpi=180, bbox_inches='tight')
fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_hflat_full_spectrum_reference.pdf', bbox_inches='tight')
plt.show()

print(f'n_half = {n_half}')
print(f'E_half = {E_half:.6f}')


## 2. Charge disorder in the perfect-correction steady state

Four complementary tests:

1. **Tr(G) histograms** at several cycles: if `perfect_correction` genuinely mixes number sectors, the per-sample distribution should be broad and need not remain centred at zero.
2. **Charge variance vs cycle**: compare `postselect` and `perfect_correction` for the `maxmix` runs.
3. **Energy vs cycle**: compare the final energy offset between protocols for both `maxmix` and `default` initial states.
4. **Occupation inference from the full spectrum**: for the `default` run, match each final-sample energy to the nearest cumulative many-body energy of the full flattened spectrum and report occupation relative to half-filling.


In [ ]:
# ── Tr(G) histograms at multiple cycles ────────────────────────────────────
CHECK_CYCLES_FRAC = [0.1, 0.5, 1.0]  # 10%, 50%, 100% of max cycle

mm_pc = run_map[('maxmix', 'perfect_correction')]
df_pc = run_map[('default', 'perfect_correction')]
cycles_arr = mm_pc['cycles']  # same for all runs
max_cycle = int(cycles_arr[-1])

check_cycle_vals = [int(f * max_cycle) for f in CHECK_CYCLES_FRAC]
# find actual saved indices
check_indices = [np.argmin(np.abs(cycles_arr - cv)) for cv in check_cycle_vals]

fig, axes = plt.subplots(len(check_indices), 2,
                          figsize=(12, 3.5 * len(check_indices)),
                          constrained_layout=True, sharey='row')

for row_i, (ci, cval) in enumerate(zip(check_indices, check_cycle_vals)):
    actual_cycle = int(cycles_arr[ci])
    for col_i, (rec, init_label) in enumerate([
        (mm_pc, 'maxmix'), (df_pc, 'default')
    ]):
        ax = axes[row_i, col_i]
        vals = rec['trace_by_sample'][:, ci]
        bins = np.arange(int(vals.min()) - 1, int(vals.max()) + 2) - 0.5
        if len(bins) < 4:
            bins = 15
        ax.hist(vals, bins=bins, color=PC_COLOR, alpha=0.75, edgecolor='black', linewidth=0.5)
        ax.axvline(0, color='k', lw=1.0, ls='-', label='Tr(G)=0 reference')
        ax.axvline(vals.mean(), color='red', lw=1.5, ls='--',
                   label=f'mean={vals.mean():.2f}')
        ax.set_title(f'init={init_label}, cycle {actual_cycle}  '
                     f'(std={vals.std():.2f})')
        ax.set_xlabel('Tr(G) per sample')
        ax.set_ylabel('count')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.2)

fig.suptitle(f'{manifest["campaign_id"]}: Tr(G) histograms — perfect_correction',
             fontsize=13)

fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_trG_histograms.png', dpi=180, bbox_inches='tight')
fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_trG_histograms.pdf', bbox_inches='tight')
plt.show()
print(f'maxmix PC final: mean={mm_pc["trace_mean"][-1]:.3f}, std={mm_pc["trace_std"][-1]:.3f}')
print(f'default PC final: mean={df_pc["trace_mean"][-1]:.3f}, std={df_pc["trace_std"][-1]:.3f}')


In [ ]:
# ── Percent deviation from half-filling histograms ─────────────────────────
# pct_dev = Tr(G) / NLAYER * 100
# (Tr(G)=0 at half-filling, so this is directly the percent filling imbalance)

fig, axes = plt.subplots(len(check_indices), 2,
                          figsize=(12, 3.5 * len(check_indices)),
                          constrained_layout=True, sharey='row')

for row_i, (ci, cval) in enumerate(zip(check_indices, check_cycle_vals)):
    actual_cycle = int(cycles_arr[ci])
    for col_i, (rec, init_label) in enumerate([
        (mm_pc, 'maxmix'), (df_pc, 'default')
    ]):
        ax = axes[row_i, col_i]
        pct_dev = rec['trace_by_sample'][:, ci] / NLAYER * 100
        pct_mean = pct_dev.mean()
        pct_std  = pct_dev.std()

        ax.hist(pct_dev, bins=20, color=PC_COLOR, alpha=0.75,
                edgecolor='black', linewidth=0.5)
        ax.axvline(0.0, color='k', lw=1.0, ls='-', label='0% reference')
        ax.axvline(pct_mean, color='red', lw=1.5, ls='--',
                   label=f'mean={pct_mean:.3f}%')

        ax.text(0.97, 0.95,
                f'$\\mu$ = {pct_mean:.3f}%\n$\\sigma$ = {pct_std:.3f}%',
                transform=ax.transAxes,
                ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.85))

        ax.set_title(f'init={init_label}, cycle {actual_cycle}')
        ax.set_xlabel('% deviation from half-filling  [Tr(G) / NLAYER × 100]')
        ax.set_ylabel('count')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.2)

fig.suptitle(
    f'{manifest["campaign_id"]}: percent deviation from half-filling — perfect_correction',
    fontsize=13,
)

fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_pct_deviation_halfilling_histograms.png',
            dpi=180, bbox_inches='tight')
fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_pct_deviation_halfilling_histograms.pdf',
            bbox_inches='tight')
plt.show()

print(f'maxmix PC final:  mean={mm_pc["trace_mean"][-1]/NLAYER*100:.4f}%,'
      f'  std={mm_pc["trace_std"][-1]/NLAYER*100:.4f}%')
print(f'default PC final: mean={df_pc["trace_mean"][-1]/NLAYER*100:.4f}%,'
      f'  std={df_pc["trace_std"][-1]/NLAYER*100:.4f}%')


In [ ]:
# ── Charge variance vs cycle (maxmix only, both protocols) ────────────────
# CV = (Nlayer - Tr(G²)) / 4
mm_ps = run_map[('maxmix', 'postselect')]

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)

# postselect: single trajectory (no std)
ax.plot(mm_ps['cycles'], mm_ps['cv_mean'],
        color=PS_COLOR, lw=2.0, label='postselect (maxmix)', **MM_STYLE)

# perfect_correction: mean ± stderr
cv_mean   = mm_pc['cv_mean']
cv_stderr = mm_pc['cv_stderr']
ax.plot(mm_pc['cycles'], cv_mean,
        color=PC_COLOR, lw=2.0, label='perfect_correction (maxmix)', **MM_STYLE)
fin = np.isfinite(cv_stderr)
if fin.any():
    ax.fill_between(mm_pc['cycles'][fin],
                    cv_mean[fin] - cv_stderr[fin],
                    cv_mean[fin] + cv_stderr[fin],
                    color=PC_COLOR, alpha=0.18)

# reference lines
ax.axhline(0.5, color='grey', lw=1.0, ls='--', label='CV=0.5 reference')
ax.axhline(0.0, color='black', lw=0.8, alpha=0.4, label='CV=0 reference')

ax.set_xlabel('Markov cycle')
ax.set_ylabel('charge variance  CV = (Nlayer − Tr(G²))/4')
ax.set_title(f'{manifest["campaign_id"]}: charge variance vs cycle (maxmix init)')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_charge_variance_vs_cycle.png', dpi=180, bbox_inches='tight')
fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_charge_variance_vs_cycle.pdf', bbox_inches='tight')
plt.show()
print(f'postselect   CV_final = {mm_ps["cv_mean"][-1]:.4f}')
print(f'perfect_corr CV_final = {mm_pc["cv_mean"][-1]:.4f}')
print(f'implied 4×CV values: postselect={4*mm_ps["cv_mean"][-1]:.2f}, '
      f'perfect_corr={4*mm_pc["cv_mean"][-1]:.2f}')


In [ ]:
# ── Flattened energy vs cycle — all four runs ──────────────────────────────
df_ps = run_map[('default', 'postselect')]

observed_gap = (run_map[('maxmix','postselect')]['energy_mean'][-1]
                - run_map[('maxmix','perfect_correction')]['energy_mean'][-1])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8),
                          constrained_layout=True, sharey=True)

for ax, init_mode, label in zip(
    axes,
    ['maxmix', 'default'],
    ['maxmix init (G=0)', 'default init (GS)']
):
    rec_pc = run_map[(init_mode, 'perfect_correction')]
    rec_ps = run_map[(init_mode, 'postselect')]

    ax.plot(rec_ps['cycles'], rec_ps['energy_mean'],
            color=PS_COLOR, lw=2.0, label='postselect', **MM_STYLE)

    e_mean   = rec_pc['energy_mean']
    e_stderr = rec_pc['energy_stderr']
    ax.plot(rec_pc['cycles'], e_mean,
            color=PC_COLOR, lw=2.0, label='perfect_correction', **MM_STYLE)
    fin = np.isfinite(e_stderr)
    if fin.any():
        ax.fill_between(rec_pc['cycles'][fin],
                        e_mean[fin] - e_stderr[fin],
                        e_mean[fin] + e_stderr[fin],
                        color=PC_COLOR, alpha=0.18)

    ax.axhline(E_half, color='grey', lw=0.9, ls='--', alpha=0.7, label='full-spectrum E_half')

    # annotate gap on maxmix panel
    if init_mode == 'maxmix':
        e_ps_final = rec_ps['energy_mean'][-1]
        e_pc_final = rec_pc['energy_mean'][-1]
        mid_y = 0.5 * (e_ps_final + e_pc_final)
        ax.annotate(
            '',
            xy=(rec_ps['cycles'][-1], e_ps_final),
            xytext=(rec_ps['cycles'][-1], e_pc_final),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5),
        )
        ax.text(rec_ps['cycles'][-1] * 0.86, mid_y,
                f'ΔE={observed_gap:.1f}',
                ha='right', va='center', fontsize=8.5,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

    ax.set_title(f'{label}')
    ax.set_xlabel('Markov cycle')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

axes[0].set_ylabel('flattened energy  E = 0.5 Tr(H G)')
fig.suptitle(f'{manifest["campaign_id"]}: flattened energy vs cycle', fontsize=13)

fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_energy_vs_cycle.png', dpi=180, bbox_inches='tight')
fig.savefig(FIGURE_ROOT / f'{manifest["campaign_id"]}_energy_vs_cycle.pdf', bbox_inches='tight')
plt.show()

print(f'Observed ΔE (postselect − perfect_correction) = {observed_gap:.2f}')
print(f'Full-spectrum half-filling reference E_half = {E_half:.2f}')


In [ ]:
# ── Inferred many-body occupation relative to half-filling — default init, full ensemble ──
# Uses the FULL flattened spectrum, with no DW/bulk separation.
# Treat every (sample, cycle) point with cycle > 5 as part of one ensemble.

df_pc = run_map[('default', 'perfect_correction')]
df_ps = run_map[('default', 'postselect')]

CYCLE_MIN = 5

# Build the default perfect_correction ensemble from all saved cycles > CYCLE_MIN
cycle_mask_pc = np.asarray(df_pc['cycles']) > CYCLE_MIN
cycles_sel_pc = np.asarray(df_pc['cycles'])[cycle_mask_pc]

tr_pc_mat = np.asarray(df_pc['trace_by_sample'])[:, cycle_mask_pc]
en_pc_mat = np.asarray(df_pc['energy_by_sample'])[:, cycle_mask_pc]

n_samples_pc, n_cycles_pc = tr_pc_mat.shape

tr_ens = tr_pc_mat.reshape(-1)
en_ens = en_pc_mat.reshape(-1)
cycle_ens = np.tile(cycles_sel_pc, n_samples_pc)
sample_ens = np.repeat(np.arange(n_samples_pc), n_cycles_pc)

# For each ensemble point, find the nearest cumulative many-body energy
dist = np.abs(cum_energy[:, None] - en_ens[None, :])
closest_idx = np.argmin(dist, axis=0)

n_closest = n_occ_grid[closest_idx]
delta_N = n_closest - n_half
e_match = cum_energy[closest_idx]
delta_E = en_ens - e_match

# Build the default postselect ensemble from all cycles > CYCLE_MIN
cycle_mask_ps = np.asarray(df_ps['cycles']) > CYCLE_MIN
cycles_sel_ps = np.asarray(df_ps['cycles'])[cycle_mask_ps]

if 'trace_by_sample' in df_ps:
    tr_ps = np.asarray(df_ps['trace_by_sample'])[:, cycle_mask_ps].reshape(-1)
else:
    tr_ps = np.asarray(df_ps['trace_mean'])[cycle_mask_ps]

if 'energy_by_sample' in df_ps:
    en_ps = np.asarray(df_ps['energy_by_sample'])[:, cycle_mask_ps].reshape(-1)
else:
    en_ps = np.asarray(df_ps['energy_mean'])[cycle_mask_ps]

dist_ps = np.abs(cum_energy[:, None] - en_ps[None, :])
closest_idx_ps = np.argmin(dist_ps, axis=0)

n_closest_ps = n_occ_grid[closest_idx_ps]
delta_N_ps = n_closest_ps - n_half
e_match_ps = cum_energy[closest_idx_ps]
delta_E_ps = en_ps - e_match_ps

print(f'n_half = {n_half}')
print(f'E_half = {E_half:.6f}')
print(f'Using default perfect_correction ensemble: cycles > {CYCLE_MIN}')
print(f'  selected cycles = {cycles_sel_pc.tolist()}')
print(f'  ensemble size    = {len(en_ens)} points ({n_samples_pc} samples x {n_cycles_pc} cycles)')
print()

for i, (cyc, samp, tr_val, e_full, n_mb, dN, e_fit, dE) in enumerate(
    zip(cycle_ens, sample_ens, tr_ens, en_ens, n_closest, delta_N, e_match, delta_E)
):
    print(
        f'pt {i:04d}: '
        f'cycle={int(cyc):3d}, '
        f'sample={int(samp):3d}, '
        f'Tr(G)={tr_val:+.3f}, '
        f'E_full={e_full:+.6f}, '
        f'n_mb_closest={n_mb:4d}, '
        f'delta_N={dN:+4d}, '
        f'E_match={e_fit:+.6f}, '
        f'delta_E={dE:+.3e}'
    )

print()
for i, (cyc, tr_val, e_full, n_mb, dN, e_fit, dE) in enumerate(
    zip(cycles_sel_ps, tr_ps, en_ps, n_closest_ps, delta_N_ps, e_match_ps, delta_E_ps)
):
    print(
        f'postselect {i:03d}: '
        f'cycle={int(cyc):3d}, '
        f'Tr(G)={tr_val:+.3f}, '
        f'E_full={e_full:+.6f}, '
        f'n_mb_closest={n_mb:4d}, '
        f'delta_N={dN:+4d}, '
        f'E_match={e_fit:+.6f}, '
        f'delta_E={dE:+.3e}'
    )

fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)

# Discrete cumulative many-body energies from the full spectrum
ax.scatter(
    n_occ_grid - n_half,
    cum_energy,
    s=16,
    facecolors='none',
    edgecolors='k',
    linewidths=0.8,
    alpha=0.8,
    label='cumulative flatband energies',
)

# Perfect-correction ensemble: all samples x all cycles > CYCLE_MIN
ax.scatter(
    delta_N,
    en_ens,
    s=18,
    color=PC_COLOR,
    alpha=0.45,
    edgecolors='none',
    label=f'default perfect_corr ensemble (n={len(en_ens)})',
)

# Postselect ensemble: all cycles > CYCLE_MIN
ax.scatter(
    delta_N_ps,
    en_ps,
    s=70,
    marker='*',
    color=PS_COLOR,
    zorder=5,
    label=f'default postselect cycles> {CYCLE_MIN} (n={len(en_ps)})',
)

ax.axvline(0, color='grey', lw=0.9, ls=':', alpha=0.7, label='half-filling')
ax.axhline(E_half, color='grey', lw=0.9, ls='--', alpha=0.7, label='E_half')

ax.set_xlabel('inferred many-body occupation relative to half-filling  ($\\Delta N$)')
ax.set_ylabel('full flattened energy')
ax.set_title(
    f"{manifest['campaign_id']}\n"
    f"Inferred many-body occupation from cumulative flatband spectrum "
    f"(default init, all samples and cycles > {CYCLE_MIN})"
)
ax.legend(fontsize=8)
ax.grid(alpha=0.2)

fig.savefig(
    FIGURE_ROOT / f"{manifest['campaign_id']}_default_ensemble_scatter_deltaN_vs_full_energy.png",
    dpi=180,
    bbox_inches='tight',
)
fig.savefig(
    FIGURE_ROOT / f"{manifest['campaign_id']}_default_ensemble_scatter_deltaN_vs_full_energy.pdf",
    bbox_inches='tight',
)
plt.show()


## Summary

| Observable | postselect | perfect_correction | Interpretation |
|---|---|---|---|
| Tr(G) | tightly constrained reference trajectory | broad sample-to-sample spread | `perfect_correction` mixes charge sectors |
| Charge variance CV | protocol reference curve | distinct steady-state curve | the two protocols end in different ensembles |
| Flattened energy | lower final energy | higher final energy | `perfect_correction` does not relax to the same half-filled reference |
| Full-spectrum occupation inference (`default`) | single reference point | broad sample cloud in `\Delta N` | the final ensemble explores occupations shifted away from half-filling |

**Interpretation**: `perfect_correction` drives the system into a mixed-sector ensemble. The full flattened spectrum provides a clean half-filling benchmark: sort the entire spectrum, build the cumulative many-body energy curve, and define half-filling by occupying all negative-energy levels. Relative to that benchmark, the `default`-init `perfect_correction` samples populate a spread of inferred many-body occupations rather than collapsing to the half-filled reference.

This notebook no longer relies on a DW-only threshold or in-gap mode count. The occupation inference is intentionally agnostic about whether the relevant occupied levels are geometrically associated with the domain wall or the bulk; it only asks where each sample energy sits relative to the cumulative many-body ladder of the full flattened Hamiltonian.

**Open question on central charge**: the Cardy-Calabrese data (separate dataset) shows different c for the two protocols. A shift in occupation relative to half-filling can change energy and charge statistics, but by itself does not explain a different c for a free chiral fermion. Possible mechanisms worth investigating:
- The `perfect_correction` steady state is a *mixed* (not pure) state; entanglement entropy of mixed states does not follow the same Cardy-Calabrese formula.
- The charge non-conservation generates off-diagonal coherences in `G` that break the assumptions of the standard free-fermion entanglement formulas.
- The effective field theory of the adaptive circuit differs from a simple free-fermion CFT; the measurement-induced transition could define a different universality class.
